# Crop Disease Classification — DenseNet121 (Transfer Learning)

**IEEE-style research pipeline — single-model notebook**

This notebook trains and evaluates **DenseNet121** (ImageNet pretrained) for multiclass
crop disease classification. The workflow, hyperparameters, augmentation, optimizer,
callbacks, epochs, and batch size are **identical across all five model notebooks** in
this study, so that the final comparison is fair. Only the backbone architecture (and
its recommended input resolution) differ — to reuse this notebook for another backbone,
change only `MODEL_NAME`, the model import, and `IMG_SIZE` in the **Configuration** and
**Load Pretrained Model** sections.

**Workflow**
1. Import libraries
2. Configuration
3. Load dataset
4. Preprocessing
5. Load pretrained model
6. Build model
7. Compile
8. Train
9. Evaluate
10. Calculate metrics
11. Save results

> Explainability / diagnostic analyses (Confusion Matrix, ROC-AUC, Grad-CAM, SHAP,
> LIME, Cross-Validation, training-history plots, etc.) are intentionally **excluded**
> here. They will be produced only for the top-3 models after the five-model comparison.


# Drive Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Import Libraries

In [2]:
# Core libraries
import os
import json
import random
import time
import numpy as np
import tensorflow as tf

# Pretrained backbone + its matching preprocessing function
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input

# Model building blocks
from tensorflow.keras import layers, models, optimizers, callbacks

# Metrics (computed manually — no plots/visual explainability in this notebook)
from sklearn.metrics import precision_score, recall_score, f1_score

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration

All hyperparameters below are **kept identical for every model** in this study so the final comparison is fair. Only `MODEL_NAME` and `IMG_SIZE` change per notebook.

> **Dataset assumption:** the dataset is organized in the standard Keras `image_dataset_from_directory` layout — one folder per split, one sub-folder per class:
> ```
> dataset/
>   train/<class_name>/*.jpg
>   val/<class_name>/*.jpg
>   test/<class_name>/*.jpg
> ```
> Update `DATASET_DIR` below to match your dataset location (e.g. a Google Drive path if running on Colab, so results persist across notebooks).

In [3]:
# ---- Identifying info (the ONLY block that changes between backbone notebooks) ----
MODEL_NAME = "DenseNet121"
IMG_SIZE   = (224, 224)   # DenseNet121's recommended input size

# ---- Reproducibility: fix random seeds for Python, NumPy and TensorFlow ----
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Paths (EDIT DATASET_DIR to match your environment) ----
DATASET_DIR = "/content/drive/MyDrive/Glaucoma_Data"                       # must contain train/ val/ test/ subfolders
RESULTS_ROOT = "Results"                                # shared root read by the comparison notebook
RESULTS_DIR = os.path.join(RESULTS_ROOT, MODEL_NAME)    # model-specific results folder: Results/<MODEL_NAME>/
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Shared hyperparameters (IDENTICAL across all 5 model notebooks) ----
BATCH_SIZE    = 32
EPOCHS        = 30
LEARNING_RATE = 1e-4
DROPOUT_RATE  = 0.3
PATIENCE      = 5          # early stopping patience


## 3. Load Dataset

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)


Found 947 files belonging to 2 classes.
Found 135 files belonging to 2 classes.
Found 273 files belonging to 2 classes.
Classes: ['Glaucoma', 'Normal']
Number of classes: 2


## 4. Preprocessing

Identical augmentation pipeline for every model: random flip, rotation, and zoom applied only to the training set, followed by the backbone's matching `preprocess_input` normalization (applied to all splits).

In [5]:
# Data augmentation — identical across all 5 model notebooks
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

def prepare(ds, training=False):
    # Apply augmentation only on the training split
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    # Apply backbone-specific preprocessing (normalization) on every split
    ds = ds.map(lambda x, y: (preprocess_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds_prepared = prepare(train_ds, training=True)
val_ds_prepared   = prepare(val_ds,   training=False)
test_ds_prepared  = prepare(test_ds,  training=False)


## 5. Load Pretrained Model (DenseNet121, ImageNet Weights)

In [6]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,),
)

# Freeze the backbone: train only the new classification head (feature extraction)
base_model.trainable = False

print(f"{MODEL_NAME} base model loaded with {len(base_model.layers)} layers (frozen).")


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
DenseNet121 base model loaded with 427 layers (frozen).


## 6. Build Model

Identical classification head architecture for every model: Global Average Pooling → BatchNormalization → Dropout → Dense (softmax).

In [7]:
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs, name=f"{MODEL_NAME}_crop_disease_classifier")
model.summary()

# Parameter counts (recorded for the JSON results file / IEEE reporting)
TOTAL_PARAMS = model.count_params()
TRAINABLE_PARAMS = int(sum(tf.keras.backend.count_params(w) for w in model.trainable_weights))
print(f"Total parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,}")


Model: "DenseNet121_crop_disease_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,043,650 (26.87 MB)

 Trainable params: 4,098 (16.01 KB)

 Non-trainable params: 7,039,552 (26.85 MB)

Total parameters:     7,043,650
Trainable parameters: 4,098


## 7. Compile

In [8]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


## 8. Train

Identical callbacks for every model: EarlyStopping + ReduceLROnPlateau + ModelCheckpoint (best weights kept, no TensorBoard / history plots in this notebook). Total training time is measured for IEEE reporting.

In [9]:
checkpoint_path = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_checkpoint.keras")

callback_list = [
    callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
]

training_start_time = time.time()

history = model.fit(
    train_ds_prepared,
    validation_data=val_ds_prepared,
    epochs=EPOCHS,
    callbacks=callback_list,
    verbose=1,
)

training_end_time = time.time()
TRAINING_TIME_SECONDS = training_end_time - training_start_time
print(f"Total training time: {TRAINING_TIME_SECONDS:.2f} seconds "
      f"({TRAINING_TIME_SECONDS / 60:.2f} minutes)")


Epoch 1/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 307s 9s/step - accuracy: 0.4171 - loss: 1.2462 - val_accuracy: 0.4148 - val_loss: 1.7839 - learning_rate: 1.0000e-04
Epoch 2/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 23s 766ms/step - accuracy: 0.4952 - loss: 1.0016 - val_accuracy: 0.4148 - val_loss: 1.3212 - learning_rate: 1.0000e-04
Epoch 3/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 763ms/step - accuracy: 0.5639 - loss: 0.8571 - val_accuracy: 0.4370 - val_loss: 1.0701 - learning_rate: 1.0000e-04
Epoch 4/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 27s 883ms/step - accuracy: 0.6125 - loss: 0.7924 - val_accuracy: 0.4667 - val_loss: 0.9307 - learning_rate: 1.0000e-04
Epoch 5/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 23s 753ms/step - accuracy: 0.6410 - loss: 0.7246 - val_accuracy: 0.4741 - val_loss: 0.8472 - learning_rate: 1.0000e-04
Epoch 6/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 25s 789ms/step - accuracy: 0.6631 - loss: 0.6643 - val_accuracy: 0.5037 - val_loss: 0.7843 - learning_rate: 1.0000e-04
Epoch 7/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 41s 781ms/step - accur

## 9. Evaluate

In [10]:
test_loss, test_accuracy = model.evaluate(test_ds_prepared, verbose=1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


9/9 ━━━━━━━━━━━━━━━━━━━━ 106s 12s/step - accuracy: 0.8425 - loss: 0.3659
Test Loss: 0.3659
Test Accuracy: 0.8425


## 10. Calculate Metrics

In [11]:
# Best train/val accuracy achieved during training (identical protocol for all models)
train_accuracy = float(max(history.history["accuracy"]))
val_accuracy   = float(max(history.history["val_accuracy"]))

# Predict on the test set for Precision / Recall / F1 (macro-averaged, multiclass)
y_true, y_pred = [], []
for images, labels in test_ds_prepared:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall    = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1        = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Train Accuracy:      {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Test Accuracy:       {test_accuracy:.4f}")
print(f"Precision (macro):   {precision:.4f}")
print(f"Recall (macro):      {recall:.4f}")
print(f"F1-score (macro):    {f1:.4f}")
print(f"Test Loss:           {test_loss:.4f}")


Train Accuracy:      0.8099
Validation Accuracy: 0.8296
Test Accuracy:       0.8425
Precision (macro):   0.8383
Recall (macro):      0.8375
F1-score (macro):    0.8379
Test Loss:           0.3659


## 11. Save Results

Saves the best trained model (`.keras`) and a JSON results file into the model-specific folder `Results/<MODEL_NAME>/`. The final comparison notebook reads every `Results/*/*_results.json` file.

In [12]:
# ---- Save the best Keras model ----
best_model_path = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_best.keras")
model.save(best_model_path)
print(f"Best model saved to: {best_model_path}")

# ---- Build the JSON results record ----
results = {
    "model": MODEL_NAME,
    "train_accuracy": train_accuracy,
    "val_accuracy": val_accuracy,
    "test_accuracy": float(test_accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "test_loss": float(test_loss),
    "training_time": TRAINING_TIME_SECONDS,
    "total_parameters": TOTAL_PARAMS,
    "trainable_parameters": TRAINABLE_PARAMS,
    "image_size": list(IMG_SIZE),
    "batch_size": BATCH_SIZE,
    "epochs": len(history.epoch),      # actual epochs completed (early stopping aware)
    "learning_rate": LEARNING_RATE,
}

results_path = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to: {results_path}")
results


Best model saved to: Results/DenseNet121/DenseNet121_best.keras
Results saved to: Results/DenseNet121/DenseNet121_results.json


{'model': 'DenseNet121',
 'train_accuracy': 0.8099260926246643,
 'val_accuracy': 0.8296296000480652,
 'test_accuracy': 0.8424908518791199,
 'precision': 0.8383296460176992,
 'recall': 0.837471036080768,
 'f1_score': 0.8378882244900777,
 'test_loss': 0.36585840582847595,
 'training_time': 1186.147183895111,
 'total_parameters': 7043650,
 'trainable_parameters': 4098,
 'image_size': [224, 224],
 'batch_size': 32,
 'epochs': 30,
 'learning_rate': 0.0001}